# Scrollytelling Data Aggregation

This notebook reads `NammaMetro_Ridership_Dataset.csv`, filters to the editorial window (2024-10-26 to 2025-05-05), and emits 5 JSON files into `scrolly-data/` for the scrollytelling web app to consume.

The web app does no data wrangling — it just reads JSONs and renders.

## Outputs

1. `daily-by-mode.json` — daily ridership per payment channel + total
2. `mode-shares.json` — same shape, each field as a fraction of the day
3. `significant-events.json` — events from `significant_dates.csv` filtered to the window
4. `anomalies.json` — days where any channel's value diverges from its 7-day moving average by >30%
5. `stations.geojson` — station data from `NammaMetro_Stations_Dataset.csv` (copied)

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

DATA_DIR = Path('/Users/home/DEV/MY PROJECTS/namma-metro-ridership-tracker')
OUT_DIR = DATA_DIR / 'scrolly-article/'
OUT_DIR.mkdir(parents=True, exist_ok=True)

EDITORIAL_START = pd.Timestamp('2024-11-14')
EDITORIAL_END = pd.Timestamp('2025-04-16')

df = pd.read_csv(DATA_DIR / 'NammaMetro_Ridership_Dataset.csv')
df['date'] = pd.to_datetime(df['Record Date'], format='%d-%m-%Y')
df = df.sort_values('date').reset_index(drop=True)
print(f'Loaded {len(df)} rows: {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Editorial window: {EDITORIAL_START.date()} → {EDITORIAL_END.date()}')

window = df[(df['date'] >= EDITORIAL_START) & (df['date'] <= EDITORIAL_END)].copy()
print(f'Rows in window: {len(window)}')

Loaded 438 rows: 2024-10-26 → 2026-07-15
Editorial window: 2024-11-14 → 2025-04-16
Rows in window: 123


## 1. Daily ridership by mode

In [ ]:
CHANNELS = {
    'Total Smart Cards': 'smartcard',
    'Total Tokens': 'token',
    'Total NCMC': 'ncmc',
    'Group Ticket': 'groupTicket',
    'QR NammaMetro': 'qrNammaMetro',
    'QR WhatsApp': 'qrWhatsApp',
    'QR Paytm': 'qrPaytm',
}

by_mode = pd.DataFrame({'date': window['date'].dt.strftime('%Y-%m-%d')})
for src, dst in CHANNELS.items():
    by_mode[dst] = window[src].astype(int).values
by_mode['total'] = window[list(CHANNELS)].sum(axis=1).astype(int).values

by_mode.to_json(OUT_DIR / 'daily-by-mode.json', orient='records', indent=2)
print(f'Wrote daily-by-mode.json: {len(by_mode)} rows')
by_mode.tail()

## 2. Mode shares per day

Each channel's share of the day's total ridership. Useful for the stacked-area chart in chapter 2 ("The seven doors") and chapter 8 ("Fare Hike Impact").

In [ ]:
shares = by_mode.copy()
for src, dst in CHANNELS.items():
    shares[dst] = (shares[dst] / shares['total']).round(4)
shares = shares.drop(columns=['total'])

shares.to_json(OUT_DIR / 'mode-shares.json', orient='records', indent=2)
print(f'Wrote mode-shares.json: {len(shares)} rows')
shares.head()

## 3. Significant events

Events from `significant_dates.csv` filtered to the editorial window. Categorised by type: `fare_hike`, `festival`, `civic`, `concert`, `bandh`, `sports`.

In [ ]:
events = pd.read_csv(DATA_DIR / 'significant_dates.csv')
events['date'] = pd.to_datetime(events['Date'], format='%Y-%m-%d')
events = events[(events['date'] >= EDITORIAL_START) & (events['date'] <= EDITORIAL_END)].copy()

def classify(label: str) -> tuple[str, str]:
    """Return (type, weight) for a given event label."""
    l = label.lower()
    if 'fare hike' in l:
        return 'fare_hike', 'primary'
    if 'concert' in l:
        return 'concert', 'primary'
    if 'bandh' in l:
        return 'bandh', 'secondary'
    if 'aero india' in l:
        return 'civic', 'primary'
    if any(k in l for k in ('diwali', 'holi', 'christmas', 'eid', 'ugadi', 'sankranti', 'pongal', 'onam', 'dasara', 'dussehra', 'navratri', 'republic day', 'independence day', 'may day', 'new year', 'valentine', 'rajayotsava', 'mahashivratri', 'ganesh', 'easter', 'good friday', 'gandhi', 'basava', 'ambedkar', 'mahavir', 'kanakadasa', 'gurupadavwa', 'varalakshmi', 'muharram', 'milad', 'equinox', 'solstice', 'bandh', 'marathon', 'champions trophy', 'asia cup', 't20 world cup', 'ipl', 'ranji', 'rcb', 'cricket', 'match', 'concert', 'aero india', 'toyota', 'lunar', 'mahalaya', 'karnataka')):
        return 'festival', 'secondary'
    return 'other', 'secondary'

events[['type', 'weight']] = events['Event'].apply(lambda l: pd.Series(classify(l)))
events_out = events[['date', 'Event', 'type', 'weight']].copy()
events_out['date'] = events_out['date'].dt.strftime('%Y-%m-%d')
events_out = events_out.rename(columns={'Event': 'label'})

events_out.to_json(OUT_DIR / 'significant-events.json', orient='records', indent=2)
print(f'Wrote significant-events.json: {len(events_out)} events')
events_out.head(20)

## 4. Anomaly detection

A day is flagged as anomalous if any channel's value diverges from its 7-day moving average by more than 30%. The baseline is computed using the 7 days *before* the day in question (so the day itself doesn't contaminate the baseline). For the first 7 days in the window, we use whatever history is available.

The two-direction rule catches:
- **Drops** — open-loop payment channels failing (e.g. Jan 15 Tokens, Jan 16 QR)
- **Surges** — closed-loop channels gaining (e.g. Jan 15-16 Smart Card)

The conspiracy chapter (9) uses these to investigate the Jan 15-16 sequence.

In [ ]:
anomalies = []
for src, dst in CHANNELS.items():
    series = window[['date', src]].copy()
    series['ma7'] = series[src].rolling(window=7, min_periods=1).mean().shift(1)
    series['pct_change'] = ((series[src] - series['ma7']) / series['ma7'] * 100).round(2)
    flagged = series[series['pct_change'].abs() > 30].copy()
    for _, row in flagged.iterrows():
        anomalies.append({
            'date': row['date'].strftime('%Y-%m-%d'),
            'channel': dst,
            'value': int(row[src]),
            'baseline': round(float(row['ma7']), 1) if pd.notna(row['ma7']) else None,
            'changePct': float(row['pct_change']),
        })

anomalies_df = pd.DataFrame(anomalies).sort_values(['date', 'channel']).reset_index(drop=True)
anomalies_df.to_json(OUT_DIR / 'anomalies.json', orient='records', indent=2)
print(f'Wrote anomalies.json: {len(anomalies_df)} anomalies across the {len(CHANNELS)} channels')
print(f'\nAnomalies on Jan 15-16 2025 (the smoking gun):')
display(anomalies_df[anomalies_df['date'].isin(['2025-01-15', '2025-01-16'])])
print(f'\nAnomalies on Dec 6 2024 (the QR NammaMetro drop):')
display(anomalies_df[anomalies_df['date'] == '2024-12-06'])

## 5. Stations

Copied from `NammaMetro_Stations_Dataset.csv` and converted to GeoJSON. The line-color emoji codes are mapped to hex colors for the map chapter (1).

In [ ]:
LINE_COLORS = {
    '🟣': '#7e3eb5',  # Purple Line
    '🟢': '#4caf50',  # Green Line
    '🟡': '#fbc02d',  # Yellow Line (future)
    '🔵': '#2196f3',  # Blue Line (future)
    '🟠': '#ff9800',  # Orange Line (future)
    '🔴': '#e53935',  # Red Line (future)
}

stations = pd.read_csv(DATA_DIR / 'NammaMetro_Stations_Dataset.csv')
print(f'Loaded {len(stations)} stations')
display(stations.head(10))

# Note: the existing CSV doesn't have lat/lng — just station metadata.
# For the scrolly map, we'll use a hand-coded schematic of the two operational lines.
# This is a placeholder GeoJSON with the station list and line colors, but no coordinates.
# The actual line geometry will be hardcoded in the scrolly component.

features = []
for _, row in stations.iterrows():
    features.append({
        'type': 'Feature',
        'properties': {
            'code': row['code'],
            'name': row['name_eng'],
            'name_kan': row['name_kan'],
            'lines': list(row['lines']) if isinstance(row['lines'], str) else [],
            'lineColors': [LINE_COLORS.get(c, '#999') for c in (row['lines'] if isinstance(row['lines'], str) else [])],
            'isTerminus': bool(row['is_terminus']),
            'isInterchange': bool(row['is_interchange']),
        },
        'geometry': None,  # coordinates to be hand-coded in the scrolly component
    })

geojson = {
    'type': 'FeatureCollection',
    'features': features,
    'metadata': {
        'lines': {emoji: hex_color for emoji, hex_color in LINE_COLORS.items()},
        'note': 'Geometry is null in this file. The scrolly map (Chapter 1) hand-codes the line paths.',
    },
}

with open(OUT_DIR / 'stations.geojson', 'w') as f:
    json.dump(geojson, f, indent=2, ensure_ascii=False)
print(f'\nWrote stations.geojson: {len(features)} features')

## 6. Hypothesis-testing window (for Chapter 9)

The notebook's chapter 9 uses 'before vs after Jan 14' without specifying the window. The scrolly uses **14 days each side** of Jan 14, per sign-off. The 14-day pre-period and 14-day post-period are exported for the chapter 9 H₀/H₁ test to use directly.

In [ ]:
PIVOT1 = pd.Timestamp('2024-12-19')
PIVOT2 = pd.Timestamp('2025-01-20')
WINDOW_DAYS = 30
pre_start = PIVOT1 - pd.Timedelta(days=WINDOW_DAYS)
pre_end = PIVOT1 - pd.Timedelta(days=1)
post_start = PIVOT2 + pd.Timedelta(days=1)
post_end = PIVOT2 + pd.Timedelta(days=WINDOW_DAYS)

pre = by_mode[(by_mode['date'] >= pre_start.strftime('%Y-%m-%d')) & (by_mode['date'] <= pre_end.strftime('%Y-%m-%d'))].copy()
post = by_mode[(by_mode['date'] >= post_start.strftime('%Y-%m-%d')) & (by_mode['date'] <= post_end.strftime('%Y-%m-%d'))].copy()
print(f'Pre-event period:  {pre["date"].iloc[0]} → {pre["date"].iloc[-1]} ({len(pre)} days)')
print(f'Post-event period: {post["date"].iloc[0]} → {post["date"].iloc[-1]} ({len(post)} days)')

hypothesis_window = {
    'pivot': '2025-01-14',
    'pivotLabel': 'Jan 14, 2025 (Sankranti / day before the anomaly)',
    'windowDays': WINDOW_DAYS,
    'pre': {
        'start': pre_start.strftime('%Y-%m-%d'),
        'end': pre_end.strftime('%Y-%m-%d'),
        'days': pre['date'].tolist(),
    },
    'post': {
        'start': post_start.strftime('%Y-%m-%d'),
        'end': post_end.strftime('%Y-%m-%d'),
        'days': post['date'].tolist(),
    },
    'hypothesis': {
        'H0': 'There is no significant difference in the mean transaction volume before and after Jan 14 for any payment method. Any observed difference is due to random variation.',
        'H1': 'There is a significant difference in the mean transaction volume before and after Jan 14 for any payment method. The difference indicates a systematic shift in commuter behaviour.',
    },
}

with open(OUT_DIR / 'hypothesis-window.json', 'w') as f:
    json.dump(hypothesis_window, f, indent=2)
print(f'\nWrote hypothesis-window.json')
print(f'Pre-event days:  {len(hypothesis_window["pre"]["days"])}')
print(f'Post-event days: {len(hypothesis_window["post"]["days"])}')

## 7. Fare-hike window (for Chapter 8)

The notebook's chapter 8 uses a 6-week window centered on Feb 9. The scrolly extends to 12 weeks (3 weeks before, 12 weeks after — full data).

In [ ]:
FARE_HIKE = '2025-02-09'
FARE_HIKE_PRE_DAYS = 22
FARE_HIKE_POST_DAYS = 24

FARE_HIKE = pd.Timestamp(FARE_HIKE)
fh_pre_start = FARE_HIKE - pd.Timedelta(days=FARE_HIKE_PRE_DAYS)
fh_post_end = FARE_HIKE + pd.Timedelta(days=FARE_HIKE_POST_DAYS)
fh_pre = by_mode[(by_mode['date'] >= fh_pre_start.strftime('%Y-%m-%d')) & (by_mode['date'] < FARE_HIKE.strftime('%Y-%m-%d'))].copy()
fh_post = by_mode[(by_mode['date'] > FARE_HIKE.strftime('%Y-%m-%d')) & (by_mode['date'] <= fh_post_end.strftime('%Y-%m-%d'))].copy()
print(f'Fare-hike window: {FARE_HIKE.date()}')
print(f'Pre-event:  {fh_pre["date"].iloc[0]} → {fh_pre["date"].iloc[-1]} ({len(fh_pre)} days)')
print(f'Post-event: {fh_post["date"].iloc[0]} → {fh_post["date"].iloc[-1]} ({len(fh_post)} days)')

fare_hike_window = {
    'pivot': FARE_HIKE.strftime('%Y-%m-%d'),
    'pivotLabel': 'Feb 9, 2025 (Fare hike)',
    'preDays': FARE_HIKE_PRE_DAYS,
    'postDays': FARE_HIKE_POST_DAYS,
    'pre': {
        'start': fh_pre_start.strftime('%Y-%m-%d'),
        'end': fh_pre['date'].iloc[-1],
        'days': fh_pre['date'].tolist(),
    },
    'post': {
        'start': fh_post['date'].iloc[0],
        'end': fh_post['date'].iloc[-1],
        'days': fh_post['date'].tolist(),
    },
}

with open(OUT_DIR / 'fare-hike-window.json', 'w') as f:
    json.dump(fare_hike_window, f, indent=2)
print(f'\nWrote fare-hike-window.json')

In [ ]:
# 5 quintile bands across the dataset's reported totals.
# The web app's tooltip chart paints 5 hard-stop colour zones at
# these value-positions, so each band represents ~20% of the days
# (not 20% of the value range). This is the v0.9 percentile-based
# scheme with the tails included — the previous '5 bands across the
# absolute min–max range' caused days to bunch in the dark bands
# because the data is right-skewed. Quintile bucketing distributes
# the data evenly across the 5 bands.
totals = by_mode['total'].astype(int).values
data_min = int(totals.min())
data_max = int(totals.max())
data_mean = int(round(float(totals.mean())))
p20 = int(np.percentile(totals, 20))
p40 = int(np.percentile(totals, 40))
p60 = int(np.percentile(totals, 60))
p80 = int(np.percentile(totals, 80))

daily_stats = {
    'min': data_min,
    'max': data_max,
    'mean': data_mean,
    'count': len(totals),
    'buckets': {
        'p20': p20,
        'p40': p40,
        'p60': p60,
        'p80': p80,
    },
    'bucketLabels': {
        'p20': '20th percentile',
        'p40': '40th percentile',
        'p60': '60th percentile',
        'p80': '80th percentile',
    },
    'bucketCount': 5,
}

with open(OUT_DIR / 'daily-stats.json', 'w') as f:
    json.dump(daily_stats, f, indent=2)
print(f'Wrote daily-stats.json')
print(f'  min  = {data_min:,} = {data_min/100000:.1f}L')
print(f'  mean = {data_mean:,} = {data_mean/100000:.1f}L')
print(f'  max  = {data_max:,} = {data_max/100000:.1f}L')
print(f'  p20  = {p20:,} = {p20/100000:.1f}L')
print(f'  p40  = {p40:,} = {p40/100000:.1f}L')
print(f'  p60  = {p60:,} = {p60/100000:.1f}L')
print(f'  p80  = {p80:,} = {p80/100000:.1f}L')
print(f'  count = {len(totals)} days, 5 bands, ~{len(totals)/5:.0f} days per band')


NameError: name 'by_mode' is not defined

## 8. Done — what's in `scrolly-article/`

In [ ]:
for f in sorted(OUT_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:30s} {size_kb:8.1f} KB')